# FINS5557 Week 6 — Tech Lab: Python AI Stack for Finance
**UNSW Business School | Hric & Lin, 2026**

---

This Tech Lab consolidates the technical foundations from Weeks 1–5 through
four hands-on coding challenges. Run each section sequentially; earlier cells
define datasets and functions used by later challenges.

| Challenge | Topic | Key skill |
|---|---|---|
| **Setup** | Environment, datasets, LLM providers | Multi-provider API authentication |
| **§1** | Financial data loading and EDA | pandas, numpy, rolling windows |
| **§2** | LLM API setup (Gemini, Claude, OpenAI) | Multi-provider abstraction |
| **C1** | Credit risk classifier pipeline | sklearn Pipeline, SHAP |
| **C2** | LLM earnings report parser | Prompt engineering, JSON extraction |
| **C3** | Momentum backtesting loop | Signal generation, Sharpe/MDD |
| **C4** | Streamlit credit scoring dashboard | Streamlit, pickle, pyngrok |
| **E1** | Extension: LLM prompt chaining | 2-step extraction + commentary |

**API keys (optional):** §2 detects available keys automatically.
Add `GOOGLE_API_KEY` (free — see §2) to Colab Secrets for live LLM exercises.
All challenges include mock paths that run without any API key.


In [ ]:
# Week 6 Tech Lab -- environment setup and dataset generation
# Run this cell first; it prepares all datasets used across the four challenges.
# !pip install shap anthropic openai google-generativeai --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings, json, sys, pickle, os
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
warnings.filterwarnings('ignore')

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('shap not installed -- run: pip install shap')

plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans',
                     'axes.spines.top': False, 'axes.spines.right': False})
NAVY   = '#003366'
RED    = '#C8102E'
GREEN  = '#1A7A4A'
ORANGE = '#D4820A'
PURPLE = '#6B21A8'

# ---- Dataset A: Synthetic credit applicants (same as Week 5) ----
# 10,000 applicants, 8% default rate, traditional + alternative data features
np.random.seed(2025)
N = 10_000
credit_score      = np.clip(np.random.normal(680, 80, N), 300, 850).astype(int)
income_k          = np.clip(np.random.lognormal(4.2, 0.5, N), 25, 500)
employment_years  = np.clip(np.random.exponential(4, N), 0, 40)
debt_to_income    = np.clip(np.random.beta(2, 5, N) * 0.8, 0, 0.8)
num_credit_lines  = np.clip(np.random.poisson(3, N), 0, 15).astype(int)
previous_defaults = (np.random.rand(N) < 0.08).astype(int)
loan_amount_k     = np.clip(np.random.lognormal(3.5, 0.8, N), 1, 200)
util_payment_score = np.clip(np.random.beta(7, 2, N), 0, 1)
rental_months      = np.clip(np.random.exponential(18, N), 0, 72)
tx_velocity        = np.clip(np.random.poisson(45, N), 1, 200)
balance_stability  = np.clip(np.random.beta(4, 3, N), 0, 1)
gig_income_pct     = np.clip(np.random.beta(1, 8, N), 0, 0.6)
digital_engagement = np.clip(np.random.beta(5, 3, N), 0, 1)
logit = (
    -4.50  - 0.015 * (credit_score - 680) + 0.80 * previous_defaults
    + 2.50 * debt_to_income - 0.030 * income_k - 0.050 * employment_years
    - 0.80 * util_payment_score - 0.60 * balance_stability
    + 0.40 * gig_income_pct + np.random.normal(0, 0.3, N)
)
default = (np.random.rand(N) < 1 / (1 + np.exp(-logit))).astype(int)

TRADITIONAL = ['credit_score', 'income_k', 'employment_years', 'debt_to_income',
               'num_credit_lines', 'previous_defaults', 'loan_amount_k']
ALT_DATA    = ['util_payment_score', 'rental_months', 'tx_velocity',
               'balance_stability', 'gig_income_pct', 'digital_engagement']
ALL_FEAT    = TRADITIONAL + ALT_DATA

df_credit = pd.DataFrame({
    'credit_score': credit_score,   'income_k': income_k,
    'employment_years': employment_years, 'debt_to_income': debt_to_income,
    'num_credit_lines': num_credit_lines, 'previous_defaults': previous_defaults,
    'loan_amount_k': loan_amount_k, 'util_payment_score': util_payment_score,
    'rental_months': rental_months, 'tx_velocity': tx_velocity,
    'balance_stability': balance_stability, 'gig_income_pct': gig_income_pct,
    'digital_engagement': digital_engagement, 'default': default,
})

# ---- Dataset B: Simulated ASX price data (for backtesting challenge) ----
# 504 business days (~2 years), 5 major ASX stocks
np.random.seed(100)
DATES   = pd.date_range('2022-01-01', periods=504, freq='B')
TICKERS = ['ANZ.AX', 'CBA.AX', 'NAB.AX', 'WBC.AX', 'MQG.AX']
PRICES  = pd.DataFrame(
    np.column_stack([
        100 * np.exp(np.random.normal(0.0003, 0.015, 504).cumsum()) for _ in TICKERS
    ]),
    index=DATES, columns=TICKERS,
)

print(f'Credit dataset:  {N:,} applicants | Default rate: {default.mean():.1%}')
print(f'Price dataset:   {len(DATES)} business days x {len(TICKERS)} tickers')
print(f'Traditional features ({len(TRADITIONAL)}): {TRADITIONAL}')
print(f'Alternative data  ({len(ALT_DATA)}): {ALT_DATA}')
print('Environment ready.')


---
## §1 — Financial Data Loading and Exploratory Analysis

**Objectives:** Apply pandas datetime indexing, rolling window calculations, and
visualisation patterns standard in quantitative finance workflows.

### Key pandas patterns for finance

```python
# Datetime indexing and resampling
prices = pd.read_csv('prices.csv', index_col='Date', parse_dates=True)
monthly = prices.resample('ME').last()       # month-end prices

# Rolling statistics
returns      = prices.pct_change()
rolling_vol  = returns.rolling(20).std() * np.sqrt(252)   # annualised volatility
rolling_corr = returns['ANZ.AX'].rolling(60).corr(returns['CBA.AX'])

# Momentum signal
momentum_20d = prices.pct_change(20).shift(1)   # shift(1) prevents look-ahead
```

### DDF Station 1 mapping
The pandas operations above implement **DDF Station 1 (ETL)**:
raw price/transaction data → clean, aligned DataFrames suitable for feature engineering.


In [ ]:
# Section 1: Financial data EDA
# Visualises the credit dataset feature distributions and ASX price data patterns.

# ---- Panel 1: Credit dataset ----
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
is_def = df_credit['default'] == 1
for i, feat in enumerate(TRADITIONAL):
    axes[i].hist(df_credit.loc[~is_def, feat], bins=30, alpha=0.6,
                 color=NAVY, density=True, label='Non-default')
    axes[i].hist(df_credit.loc[ is_def, feat], bins=30, alpha=0.6,
                 color=RED,  density=True, label='Default')
    axes[i].set_title(feat, fontsize=9)
axes[7].axis('off')
axes[7].legend(handles=[mpatches.Patch(color=NAVY, label='Non-default'),
                         mpatches.Patch(color=RED,  label='Default')],
               loc='center', fontsize=10)
fig.suptitle('Credit Dataset: Feature Distributions by Default Status', color=NAVY, fontsize=12)
plt.tight_layout()
plt.show()

# ---- Panel 2: Price data ----
returns_price = PRICES.pct_change().dropna()
rolling_vol   = returns_price.rolling(20).std() * np.sqrt(252) * 100

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
palette = [NAVY, RED, GREEN, ORANGE, PURPLE]
for i, ticker in enumerate(TICKERS):
    norm = PRICES[ticker] / PRICES[ticker].iloc[0] * 100
    axes[0].plot(PRICES.index, norm, lw=1.5, label=ticker, color=palette[i])
axes[0].set_title('Normalised Price Performance (Base=100)', color=NAVY)
axes[0].legend(fontsize=8)
axes[0].set_ylabel('Indexed Price')

axes[1].plot(rolling_vol.index, rolling_vol, lw=1.2, alpha=0.85)
axes[1].set_title('20-Day Rolling Volatility (Ann. %)', color=NAVY)
axes[1].set_ylabel('Volatility (%)')
axes[1].legend(TICKERS, fontsize=7)

corr = returns_price.corr()
im = axes[2].imshow(corr.values, cmap='RdYlGn', vmin=0, vmax=1)
axes[2].set_xticks(range(len(TICKERS))); axes[2].set_xticklabels(TICKERS, rotation=30, fontsize=8)
axes[2].set_yticks(range(len(TICKERS))); axes[2].set_yticklabels(TICKERS, fontsize=8)
axes[2].set_title('Return Correlation Matrix', color=NAVY)
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
for i in range(len(TICKERS)):
    for j in range(len(TICKERS)):
        axes[2].text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.tight_layout()
plt.show()

# Summarise
print("Credit Dataset Descriptive Statistics:")
print(df_credit[TRADITIONAL + ['default']].describe().round(2).to_string())
print(f"\nASX Returns -- Annualised Sharpe (approx):")
for t in TICKERS:
    sr = returns_price[t].mean() / returns_price[t].std() * np.sqrt(252)
    print(f"  {t}: SR = {sr:.2f}")


---
## §2 — Multi-Provider LLM Setup

This lab supports three LLM providers. The cell below detects available API keys
automatically. **No key is required** — a full mock simulation runs by default.

| Provider | Free Tier | Best Model | Python Package |
|---|---|---|---|
| **Google Gemini** | Yes (15 RPM / 1M TPD) | `gemini-1.5-flash` | `google-generativeai` |
| **Anthropic Claude** | No (free credits on sign-up) | `claude-haiku-4-5-20251001` | `anthropic` |
| **OpenAI** | No (free credits on sign-up) | `gpt-4o-mini` | `openai` |

### Setup (Gemini — recommended for this lab)
1. Visit `https://aistudio.google.com/apikey` — generate a free key (no credit card)
2. In Colab: click the **key icon** (Secrets) → **+ Add new secret**
3. Name: `GOOGLE_API_KEY` | Value: paste key | Toggle **Notebook access** ON
4. Re-run §2 — the notebook auto-detects the key

### Additional providers (Colab Secrets names)
- Anthropic: `ANTHROPIC_API_KEY`
- OpenAI: `OPENAI_API_KEY`


In [ ]:
# Section 2: Multi-provider LLM detection and unified call abstraction
# Priority: Gemini (free) > Anthropic Claude > OpenAI > Mock

def _load_secret(name):
    # Load API key from Colab Secrets, then OS env, then return None.
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.getenv(name)

GEMINI_KEY    = _load_secret("GOOGLE_API_KEY")
ANTHROPIC_KEY = _load_secret("ANTHROPIC_API_KEY")
OPENAI_KEY    = _load_secret("OPENAI_API_KEY")

if   GEMINI_KEY:    LLM_PROVIDER = "gemini"
elif ANTHROPIC_KEY: LLM_PROVIDER = "anthropic"
elif OPENAI_KEY:    LLM_PROVIDER = "openai"
else:               LLM_PROVIDER = "mock"

print(f"Active LLM provider: {LLM_PROVIDER.upper()}")

# Install and initialise the active provider
if LLM_PROVIDER == "gemini":
    try:
        import google.generativeai as genai
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install",
                        "google-generativeai", "-q"], check=True)
        import google.generativeai as genai
    genai.configure(api_key=GEMINI_KEY)
    _GEMINI_MODEL = genai.GenerativeModel(
        "gemini-1.5-flash",
        generation_config={"temperature": 0,
                           "response_mime_type": "application/json"})
    print("  Gemini 1.5 Flash initialised (free tier: 15 RPM / 1M TPD)")

elif LLM_PROVIDER == "anthropic":
    try:
        import anthropic as _anthropic_lib
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "anthropic", "-q"],
                       check=True)
        import anthropic as _anthropic_lib
    _ANTHROPIC_CLIENT = _anthropic_lib.Anthropic(api_key=ANTHROPIC_KEY)
    print("  Anthropic Claude Haiku initialised")

elif LLM_PROVIDER == "openai":
    try:
        from openai import OpenAI as _OpenAIClass
    except ImportError:
        import subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "openai", "-q"],
                       check=True)
        from openai import OpenAI as _OpenAIClass
    _OPENAI_CLIENT = _OpenAIClass(api_key=OPENAI_KEY)
    print("  OpenAI GPT-4o-mini initialised")

else:
    print("  No API key found -- mock simulation active for all LLM challenges.")
    print("  Add GOOGLE_API_KEY to Colab Secrets for free live Gemini access.")


def call_llm(prompt, system="You are a financial analyst. Return only valid JSON."):
    # Unified LLM call; returns raw text response string (or None for mock).
    if LLM_PROVIDER == "gemini":
        response = _GEMINI_MODEL.generate_content(f"{system}\n\n{prompt}")
        return response.text
    elif LLM_PROVIDER == "anthropic":
        msg = _ANTHROPIC_CLIENT.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=1024, system=system,
            messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    elif LLM_PROVIDER == "openai":
        resp = _OPENAI_CLIENT.chat.completions.create(
            model="gpt-4o-mini",
            response_format={"type": "json_object"},
            messages=[{"role": "system", "content": system},
                      {"role": "user",   "content": prompt}])
        return resp.choices[0].message.content
    return None


def parse_json_response(text):
    # Strip markdown fences and parse JSON; returns dict or raises.
    if text is None:
        raise ValueError("No response (mock path)")
    clean = text.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    return json.loads(clean.strip())


print("\ncall_llm(prompt, system) available for Challenges 2 and Extension E1.")


---
## Challenge 1 — Credit Risk Classifier Pipeline

**Goal:** Build a production-ready `sklearn.Pipeline` for credit default prediction.

### DDF mapping

| DDF Station | Pipeline step |
|---|---|
| Station 1 (ETL) | `df_credit` loaded in Setup |
| Station 2 (Feature Eng.) | `StandardScaler` — normalise all features |
| Station 3 (Model Design) | `GradientBoostingClassifier` |
| Station 4 (Implementation) | AUC evaluation + SHAP adverse action notice |

### Why Pipeline matters (SR 11-7)
A `Pipeline` applies **identical transformations** in training and inference,
eliminating data leakage. Under SR 11-7, the model risk officer validates the
Pipeline as a single artefact — the scaler and model cannot be separated
without triggering a model change event.

### Deliverables
1. Train the pipeline on all 13 features
2. Evaluate: AUC, Gini, precision-recall, confusion matrix
3. SHAP waterfall plot for the highest-risk declined applicant
4. *(Extension)* Add a `ColumnTransformer` that applies WoE encoding to `credit_score`


In [ ]:
# Challenge 1: Credit Risk Classifier Pipeline
# Demonstrates the sklearn Pipeline pattern: scaler + GradientBoostingClassifier.
# SHAP section uses TreeExplainer on the fitted model step.

X = df_credit[ALL_FEAT].values
y = df_credit['default'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Build and train the Pipeline (DDF Stations 2-3)
credit_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  GradientBoostingClassifier(
                n_estimators=200, max_depth=4,
                learning_rate=0.08, min_samples_leaf=20,
                subsample=0.8, random_state=42))
])
credit_pipeline.fit(X_train, y_train)

# Evaluate (DDF Station 4)
y_prob = credit_pipeline.predict_proba(X_test)[:, 1]
auc    = roc_auc_score(y_test, y_prob)
gini   = 2 * auc - 1
y_pred = (y_prob >= 0.35).astype(int)

print("Challenge 1: Credit Risk Classifier Pipeline")
print("=" * 55)
print(f"Pipeline: {[(name, type(step).__name__) for name, step in credit_pipeline.steps]}")
print(f"Test AUC: {auc:.4f} | Gini: {gini:.4f}")
print()
print("Classification Report (threshold = 0.35):")
print(classification_report(y_test, y_pred, target_names=['Non-default', 'Default']))

# Visualisation: ROC + Confusion Matrix + Score Distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0].plot(fpr, tpr, color=NAVY, lw=2, label=f'AUC = {auc:.3f}')
axes[0].plot([0,1],[0,1],'k:',lw=0.8, label='Random')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve', color=NAVY)
axes[0].legend(fontsize=9)

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['No Default','Default']).plot(
    ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Confusion Matrix (t=0.35)', color=NAVY)

axes[2].hist(y_prob[y_test==0], bins=40, alpha=0.6, color=NAVY, density=True, label='Non-default')
axes[2].hist(y_prob[y_test==1], bins=40, alpha=0.7, color=RED,  density=True, label='Default')
axes[2].axvline(0.35, color=ORANGE, linestyle='--', lw=1.5, label='threshold')
axes[2].set_xlabel('P(default)'); axes[2].set_ylabel('Density')
axes[2].set_title(f'Score Distribution | Gini={gini:.3f}', color=NAVY)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('challenge1_pipeline.png', bbox_inches='tight')
plt.show()

# SHAP explanations
if SHAP_AVAILABLE:
    explainer  = shap.TreeExplainer(credit_pipeline.named_steps['model'])
    X_test_sc  = credit_pipeline.named_steps['scaler'].transform(X_test[:300])
    shap_vals  = explainer.shap_values(X_test_sc)
    plt.figure(figsize=(10, 5))
    shap.summary_plot(shap_vals, X_test_sc, feature_names=ALL_FEAT,
                      show=False, plot_size=(10, 5))
    plt.title('SHAP Feature Importance (Pipeline)', color=NAVY)
    plt.tight_layout()
    plt.show()
    # Adverse action notice for the highest-risk default in test set
    hr_pos    = np.argmax(y_prob * (y_test == 1))
    x_hr      = credit_pipeline.named_steps['scaler'].transform(X_test[hr_pos:hr_pos+1])
    shap_hr   = explainer.shap_values(x_hr)[0]
    ranked    = sorted(zip(ALL_FEAT, shap_hr), key=lambda t: abs(t[1]), reverse=True)
    print("\nSHAP Adverse Action Notice (NCA s124 compliant):")
    print(f"  P(default) = {y_prob[hr_pos]:.3f} -- DECLINED (threshold 0.35)")
    print("  Top contributing factors:")
    for i, (feat_name, sv) in enumerate(ranked[:5], 1):
        direction = 'INCREASES' if sv > 0 else 'REDUCES'
        print(f"  {i}. {feat_name:25s}: {direction} default risk (SHAP={sv:+.3f})")
else:
    # Fallback: built-in feature importance
    imp = credit_pipeline.named_steps['model'].feature_importances_
    ranked = sorted(zip(ALL_FEAT, imp), key=lambda t: t[1], reverse=True)
    print("\nFeature Importance (install shap for full SHAP analysis):")
    for feat_name, v in ranked[:8]:
        print(f"  {feat_name:25s}: {v:.4f}")

# Save pipeline for Streamlit challenge
with open('credit_pipeline.pkl', 'wb') as f:
    pickle.dump(credit_pipeline, f)
print("\ncredit_pipeline.pkl saved (used in Challenge 4 Streamlit app).")


---
## Challenge 2 — LLM Earnings Report Parser

**Goal:** Use an LLM (Gemini / Claude / OpenAI) to extract structured financial
metrics from a simulated ASX earnings report excerpt.

### Prompt engineering principles
1. **Role assignment:** `"You are a financial analyst extracting metrics from ASX earnings reports"`
2. **Task precision:** list exactly which fields to extract with their expected types
3. **Output format:** `"Return ONLY valid JSON, no markdown"` — prevents code fences
4. **Null handling:** `"Use null for any field not present in the text"` — prevents hallucination

### DDF Station 1 (ETL) mapping
This challenge implements the **Document Scanner → Metrics Extractor** pipeline
from the Week 4 and Week 5 case studies, but applied to a live earnings document
rather than a pre-structured report.

### Deliverables
1. Run the LLM extraction cell (mock or live depending on your API key)
2. Parse the JSON response into a `pd.DataFrame`
3. Calculate the EPS surprise: `eps_actual - eps_consensus`
4. *(Extension)* Write a second prompt that generates a 3-sentence analyst commentary
   based on the extracted metrics (see Extension E1)


In [ ]:
# Challenge 2: LLM Earnings Report Parser
# Uses call_llm() from §2. Mock output is structurally identical to live output.

import json

EARNINGS_TEXT = """
ANZ Banking Group -- Half Year Results H2 FY2024

FINANCIAL HIGHLIGHTS
Revenue (NII + Non-Interest Income): AUD 9,824 million (H2 FY2024)
Net Profit After Tax (NPAT): AUD 3,382 million (H2 FY2024), down 3% half-on-half
Earnings Per Share (EPS): AUD 1.28 (H2 FY2024), up from AUD 1.19 (H2 FY2023)
Return on Equity (ROE): 9.8% (FY2024), down from 11.2% (FY2023)
Net Interest Margin (NIM): 1.74% (H2 FY2024), down 8bp from H1 FY2024
Cost-to-Income Ratio: 48.2% (target: <45% by FY2026)
Common Equity Tier 1 (CET1): 12.30% (above APRA D-SIB minimum 10.25%)

DIVIDEND
Interim dividend: AUD 0.83 per share (fully franked)

GUIDANCE
Management expects NIM pressure in H1 FY2025 due to deposit competition.
Cost reduction targeting AUD 400M savings by FY2026.
Institutional banking and markets expected to drive revenue growth.

ANALYST CONSENSUS (prior to result)
EPS consensus: AUD 1.22 | Revenue consensus: AUD 9,650M | NPAT consensus: AUD 3,450M
"""

SYSTEM_PROMPT = "You are a financial analyst extracting structured data from ASX earnings reports. Return only valid JSON."

EXTRACTION_PROMPT = f"""Extract the following fields from the earnings report and return as JSON:
{{
  "ticker": "ASX ticker symbol",
  "period": "reporting period",
  "revenue_M": number or null,
  "npat_M": number or null,
  "eps_aud": number or null,
  "roe_pct": number or null,
  "nim_pct": number or null,
  "cet1_pct": number or null,
  "dividend_aud": number or null,
  "guidance_summary": "one sentence",
  "eps_consensus_aud": number or null,
  "revenue_consensus_M": number or null,
  "npat_consensus_M": number or null
}}
Return ONLY valid JSON.

Earnings report:
{EARNINGS_TEXT}"""

# Mock output (structurally identical to live extraction)
MOCK_EXTRACTION = {
    "ticker": "ANZ.AX", "period": "H2 FY2024",
    "revenue_M": 9824, "npat_M": 3382, "eps_aud": 1.28,
    "roe_pct": 9.8, "nim_pct": 1.74, "cet1_pct": 12.30,
    "dividend_aud": 0.83,
    "guidance_summary": "NIM pressure expected H1 FY2025; AUD 400M cost reduction by FY2026.",
    "eps_consensus_aud": 1.22, "revenue_consensus_M": 9650, "npat_consensus_M": 3450
}

# Execute extraction
if LLM_PROVIDER != "mock":
    try:
        raw = call_llm(EXTRACTION_PROMPT, system=SYSTEM_PROMPT)
        extracted = parse_json_response(raw)
        print(f"Live {LLM_PROVIDER.upper()} extraction successful.")
    except Exception as e:
        print(f"LLM error ({e}) -- using mock output.")
        extracted = MOCK_EXTRACTION
else:
    extracted = MOCK_EXTRACTION
    print("Mock simulation active.")

# Display structured results
df_ext = pd.DataFrame([extracted]).T.rename(columns={0: 'Value'})
print("\nExtracted Earnings Metrics:")
print(df_ext.to_string())

# EPS Surprise analysis
eps_actual    = extracted.get('eps_aud')    or 0
eps_consensus = extracted.get('eps_consensus_aud') or 0
npat_actual   = extracted.get('npat_M')    or 0
npat_cons     = extracted.get('npat_consensus_M') or 0

if eps_consensus:
    surprise_abs = eps_actual - eps_consensus
    surprise_pct = surprise_abs / eps_consensus * 100
    signal = 'BEAT' if surprise_abs > 0 else ('MISS' if surprise_abs < 0 else 'IN-LINE')
    npat_surprise = npat_actual - npat_cons

    print(f"\nEPS Surprise Analysis:")
    print(f"  Actual EPS:    AUD {eps_actual:.2f}")
    print(f"  Consensus EPS: AUD {eps_consensus:.2f}")
    print(f"  Surprise:      {surprise_abs:+.2f} ({surprise_pct:+.1f}%)  --> {signal}")
    print(f"  NPAT surprise: AUD {npat_surprise:+.0f}M ({(npat_surprise/npat_cons*100):+.1f}%)")

# Save for Extension E1
import sys


---
## Challenge 3 — Momentum Backtesting Loop

**Goal:** Implement a simple 20-day momentum strategy on 5 ASX stocks, accounting
for transaction costs, and evaluate with the Sharpe/MDD/CR/AR metrics from Week 4.

### Strategy specification
- **Universe:** 5 simulated ASX stocks (generated in Setup)
- **Signal:** 20-day price momentum — go LONG if $p_t / p_{t-20} - 1 > 2\%$, else FLAT
- **Weights:** equal weight across LONG positions on each day
- **Transaction costs:** 10bp one-way per position change
- **Evaluation period:** full 504 business-day simulation

### Week 4 connection
The four performance metrics defined in Week 4 (Eq 4.1–4.4) are computed here:

$$SR = \frac{\bar{r}_p - r_f}{\sigma_p}\sqrt{252}, \quad
MDD = \max_{t} \frac{\text{Peak}_t - P_t}{\text{Peak}_t}$$

$$CR = \frac{V_T - V_0}{V_0}, \quad AR = (1 + CR)^{252/T} - 1$$

### Deliverables
1. Run the backtesting cell and interpret the performance table
2. Modify `SIGNAL_THRESH` (try 0.00 vs 0.05) and compare results
3. *(Extension)* Implement a 3-factor model: momentum + value (P/B proxy) + quality (ROE proxy)


In [ ]:
# Challenge 3: Momentum Backtesting Loop
# Implements a long-only momentum strategy with equal weighting.
# Modify SIGNAL_THRESH and LOOKBACK to experiment with the strategy.

LOOKBACK      = 20    # momentum lookback period (trading days)
SIGNAL_THRESH = 0.02  # minimum momentum to go LONG (2%)
COST_BPS      = 10    # one-way transaction cost in basis points

returns_px   = PRICES.pct_change().dropna()
momentum     = PRICES.pct_change(LOOKBACK).shift(1)  # shift(1) prevents look-ahead
momentum     = momentum.loc[returns_px.index].dropna()
returns_bt   = returns_px.loc[momentum.index]

# Signal and equal-weight positions
signals  = (momentum > SIGNAL_THRESH).astype(float)
n_long   = signals.sum(axis=1).replace(0, np.nan)
weights  = signals.div(n_long, axis=0).fillna(0)

# Portfolio returns (gross)
port_returns_gross = (weights * returns_bt).sum(axis=1)

# Transaction costs: 10bp * |change in weight| per stock
weight_change = weights.diff().abs().sum(axis=1)
costs         = weight_change * (COST_BPS / 10_000)
port_returns  = port_returns_gross - costs

# Buy-and-Hold benchmark (equal weight, always)
bh_returns = returns_bt.mean(axis=1)


def perf_metrics(ret, rf_annual=0.0533, label='Strategy'):
    cr  = (1 + ret).prod() - 1
    ar  = (1 + cr) ** (252 / len(ret)) - 1
    rf  = rf_annual / 252
    sr  = (ret.mean() - rf) / ret.std() * np.sqrt(252) if ret.std() > 0 else 0
    wealth  = (1 + ret).cumprod()
    peak    = wealth.expanding().max()
    mdd     = ((peak - wealth) / peak).max()
    return {'Label': label,
            'CR (%)':  round(cr  * 100, 2),
            'AR (%)':  round(ar  * 100, 1),
            'SR':      round(sr, 3),
            'MDD (%)': round(mdd * 100, 2)}


df_perf = pd.DataFrame([
    perf_metrics(port_returns, label='Momentum (net, 20d/2%)'),
    perf_metrics(bh_returns,   label='Buy-and-Hold'),
]).set_index('Label')

print("Challenge 3: Momentum Backtesting Results")
print("=" * 60)
print(df_perf.to_string())
print(f"\nStrategy active on {(signals.sum(axis=1) > 0).mean():.1%} of trading days")
print(f"Average tickers held: {signals.sum(axis=1)[signals.sum(axis=1) > 0].mean():.2f}")

# Visualisation
wealth_strat = (1 + port_returns).cumprod() * 100
wealth_bh    = (1 + bh_returns).cumprod()   * 100

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(wealth_strat.index, wealth_strat, color=RED,  lw=2,   label='Momentum (net)')
axes[0].plot(wealth_bh.index,    wealth_bh,    color=NAVY, lw=1.5, linestyle='--', label='Buy-and-Hold')
axes[0].set_title('Equity Curves (Base=100)', color=NAVY)
axes[0].set_ylabel('Portfolio Value')
axes[0].legend(fontsize=9)

axes[1].fill_between(signals.index, signals.sum(axis=1),
                     alpha=0.75, color=GREEN, step='post')
axes[1].set_title('Number of Long Positions per Day', color=NAVY)
axes[1].set_ylabel('N Long')
axes[1].set_ylim(0, len(TICKERS) + 0.5)

roll_sr = (port_returns.rolling(60).mean() /
           port_returns.rolling(60).std() * np.sqrt(252))
axes[2].plot(roll_sr.index, roll_sr, color=ORANGE, lw=1.5)
axes[2].axhline(0, color='black', lw=0.8, linestyle=':')
axes[2].axhline(1, color='green', lw=0.8, linestyle=':', label='SR=1.0')
axes[2].set_title('60-Day Rolling Sharpe Ratio', color=NAVY)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('challenge3_backtest.png', bbox_inches='tight')
plt.show()


---
## Challenge 4 — Streamlit Credit Scoring Dashboard

**Goal:** Wrap the Challenge 1 model in an interactive web application using
[Streamlit](https://streamlit.io) — the standard Python framework for rapid
AI/ML prototype interfaces in finance.

### App features
- **Sidebar:** input sliders for all 13 applicant features
- **Main panel:** predicted default probability, APPROVE/DECLINE badge,
  feature contribution bar chart
- **Model Card** (expandable): version, training data, AUC, regulatory anchors (SR 11-7)

### How to run
```bash
# Local (after installing streamlit)
pip install streamlit
streamlit run credit_app.py

# Google Colab (pyngrok tunnel)
!pip install pyngrok --quiet
!nohup streamlit run credit_app.py --server.port 8501 &
from pyngrok import ngrok
print(ngrok.connect(8501))
```

### Deliverables
1. Run the cell below to write `credit_app.py` to disk
2. Start the Streamlit server using one of the methods above
3. Test with two applicant profiles: high-risk (credit_score=450, DTI=0.70, prev_defaults=1)
   and low-risk (credit_score=780, DTI=0.15, prev_defaults=0)
4. *(Extension)* Add a **Batch Upload** tab: upload a CSV of applicants,
   score all rows, and display a histogram of predicted default probabilities


In [ ]:
# Challenge 4: Write credit_app.py and display run instructions.
# The Streamlit app loads the credit_pipeline.pkl saved in Challenge 1.

STREAMLIT_CODE = '''
import streamlit as st
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt

st.set_page_config(
    page_title="FINS5557 Credit Risk Dashboard",
    layout="wide",
)

@st.cache_resource
def load_model():
    with open("credit_pipeline.pkl", "rb") as f:
        return pickle.load(f)

pipeline = load_model()

ALL_FEAT = ["credit_score","income_k","employment_years","debt_to_income",
            "num_credit_lines","previous_defaults","loan_amount_k",
            "util_payment_score","rental_months","tx_velocity",
            "balance_stability","gig_income_pct","digital_engagement"]

# ---- Sidebar: applicant inputs ----
st.sidebar.header("Applicant Profile")
credit_score     = st.sidebar.slider("Credit Score",            300, 850, 680)
income_k         = st.sidebar.slider("Annual Income (AUD K)",    25, 500,  80)
employment_years = st.sidebar.slider("Employment Years",          0,  40,   5)
debt_to_income   = st.sidebar.slider("Debt-to-Income Ratio",   0.0, 0.8, 0.25, step=0.01)
num_credit_lines = st.sidebar.slider("Number of Credit Lines",   0,  15,   3)
previous_defaults= int(st.sidebar.selectbox("Previous Defaults", ["No (0)","Yes (1)"]).startswith("Y"))
loan_amount_k    = st.sidebar.slider("Loan Amount (AUD K)",       1, 200,  30)
util_pay         = st.sidebar.slider("Utility Payment Score",   0.0, 1.0, 0.8, step=0.01)
rental_months    = float(st.sidebar.slider("Rental Payment History (months)", 0, 72, 24))
tx_velocity      = float(st.sidebar.slider("Transaction Velocity (per month)", 1, 200, 45))
balance_stability= st.sidebar.slider("Balance Stability",       0.0, 1.0, 0.6, step=0.01)
gig_income_pct   = st.sidebar.slider("Gig Income Proportion",  0.0, 0.6, 0.05, step=0.01)
digital_engagement = st.sidebar.slider("Digital Engagement",   0.0, 1.0, 0.7, step=0.01)

features = np.array([[credit_score, income_k, employment_years, debt_to_income,
                       num_credit_lines, previous_defaults, loan_amount_k,
                       util_pay, rental_months, tx_velocity,
                       balance_stability, gig_income_pct, digital_engagement]])

prob     = pipeline.predict_proba(features)[0][1]
decision = "APPROVED" if prob < 0.35 else "DECLINED"
colour   = "#1A7A4A" if decision == "APPROVED" else "#C8102E"

# ---- Main panel ----
st.title("Credit Risk Scoring Dashboard")
st.markdown("**FINS5557 Applied AI in Finance | UNSW Business School | Hric & Lin, 2026**")
st.divider()

col1, col2, col3 = st.columns([1, 1, 1])
with col1:
    st.metric("P(Default)", f"{prob:.1%}")
with col2:
    st.metric("Decision Threshold", "35%")
with col3:
    badge = (
        f'<div style="background:{colour};padding:14px;border-radius:8px;'
        f'text-align:center;color:white;font-size:1.5em;font-weight:bold;">'
        f'{decision}</div>'
    )
    st.markdown(badge, unsafe_allow_html=True)

st.divider()

# Feature importance bar
feat_df = pd.DataFrame({"Feature": ALL_FEAT, "Input Value": features[0]})
st.subheader("Applicant Feature Summary")
st.dataframe(feat_df.set_index("Feature").T.round(3))

# Model Card
with st.expander("Model Card (SR 11-7 Documentation)"):
    st.markdown(
        "- **Model type:** GradientBoostingClassifier (sklearn Pipeline)  \n"
        "- **Features:** 7 traditional bureau + 6 alternative data  \n"
        "- **Training data:** 10,000 synthetic applicants (FINS5557, seed=2025)  \n"
        "- **Performance:** AUC approx 0.84, Gini approx 0.68  \n"
        "- **Decision threshold:** 0.35  \n"
        "- **Regulatory anchors:** SR 11-7, ASIC RG 255, NCA s124, Privacy Act 1988"
    )
'''

# Write app to disk
with open('credit_app.py', 'w', encoding='utf-8') as f:
    f.write(STREAMLIT_CODE)
print('credit_app.py written to disk.')
print()
print('Run the Streamlit dashboard:')
print()
print('  LOCAL:')
print('    pip install streamlit')
print('    streamlit run credit_app.py')
print()
print('  COLAB:')
print('    !pip install pyngrok --quiet')
print('    !nohup streamlit run credit_app.py --server.port 8501 &')
print('    from pyngrok import ngrok')
print('    print(ngrok.connect(8501))  # open the printed URL')
print()
print('Test applicant profiles:')
print('  High-risk: credit_score=450, DTI=0.70, prev_defaults=1')
print('  Low-risk:  credit_score=780, DTI=0.15, prev_defaults=0')


---
## Extension Challenges

These challenges are for students who complete C1–C4 with time remaining.
They integrate concepts across multiple weeks and require independent problem-solving.

| Challenge | Builds on | Skill |
|---|---|---|
| **E1** | C2 (LLM parser) | Prompt chaining, token-efficient workflows |
| **E2** | C3 (backtesting) | Factor models, cross-sectional signal construction |
| **E3** | C1 + C4 | Streamlit chatbot, LLM explanation, adverse action |
| **E4** | C1 + Week 5 PSI | Production monitoring dashboard |

**E1 — LLM Prompt Chaining:** implemented in the cell below.

**E2 — Factor Model Backtest:** Extend Challenge 3 to a 3-factor model:
(1) 20-day momentum, (2) negative 12-month return (value proxy), (3) min-variance
allocation. Rebalance monthly. Compare Sharpe ratios across factors.

**E3 — Credit Explainer Chatbot:** Add a Streamlit `st.chat_input` widget to `credit_app.py`.
When a user types "Why was I declined?", use `call_llm()` to generate a plain-language
explanation referencing the top-3 SHAP features. Ensure the explanation satisfies
NCA s124 adverse action notice requirements (no discriminatory language).

**E4 — PSI Monitoring Tab:** Add a second Streamlit tab to `credit_app.py`.
Allow the user to upload a CSV of new applicants. Compute PSI (Eq 5.3 from Week 5)
against the training population for each feature. Flag features with PSI > 0.20
in red and recommend retrain under APRA CPS 220.


In [ ]:
# Extension E1: LLM Prompt Chaining (2-step)
# Step 1: earnings extraction (reuses `extracted` from Challenge 2)
# Step 2: LLM generates a 3-sentence analyst commentary from the extracted metrics.

COMMENTARY_SYSTEM = "You are a senior equity analyst writing morning notes for institutional clients."

COMMENTARY_PROMPT = (
    "Based on the following extracted earnings metrics for an ASX-listed bank, "
    "write a 3-sentence analyst commentary covering: "
    "(1) earnings quality assessment, (2) primary risk factor, (3) investment recommendation. "
    "Be concise and precise. Avoid filler phrases. Do not use markdown formatting.\n\n"
    f"Metrics (JSON):\n{json.dumps(extracted, indent=2)}"
)

# Mock commentary (same structure as live output)
MOCK_COMMENTARY = (
    "ANZ's H2 FY2024 result reflects a bank navigating a structurally challenging NIM "
    "environment: while CET1 at 12.30% provides substantial capital headroom (+205bp above "
    "the APRA D-SIB floor), the 8bp half-on-half margin compression and 23% rise in credit "
    "impairments signal that the asset-quality cycle is deteriorating faster than consensus anticipated. "
    "The 0.84% mortgage arrears rate -- approaching the APRA CPS 220 1.00% supervisory trigger -- "
    "represents the key near-term monitoring risk, amplified by the NZ portfolio stress (+18% provisions). "
    "Maintain HOLD; initiate review trigger for H1 FY2025 NZ credit quality disclosure before "
    "considering an upgrade, given the approximately AUD 1.97 discount-to-book implied upside "
    "on normalised earnings."
)

print("Extension E1: LLM Prompt Chaining (2-step workflow)")
print("=" * 60)
print(f"\nStep 1 -- Source: {len(EARNINGS_TEXT)} chars earnings text")
print(f"  Extracted: ticker={extracted.get('ticker')}, EPS={extracted.get('eps_aud')}, "
      f"NPAT={extracted.get('npat_M')}M")

print(f"\nStep 2 -- Commentary generation ({LLM_PROVIDER.upper()})")
if LLM_PROVIDER != "mock":
    try:
        commentary = call_llm(COMMENTARY_PROMPT, system=COMMENTARY_SYSTEM)
        # commentary may not be JSON -- override response_mime_type for this call
        if commentary is None:
            commentary = MOCK_COMMENTARY
        print(f"  [{LLM_PROVIDER.upper()} -- {len(commentary)} chars]")
    except Exception as e:
        print(f"  LLM error ({e}) -- using mock commentary.")
        commentary = MOCK_COMMENTARY
else:
    commentary = MOCK_COMMENTARY
    print(f"  [MOCK -- {len(commentary)} chars]")

# Format output
print("\nANALYST COMMENTARY:")
print("-" * 60)
# Split into sentences for readability
sentences = [s.strip() for s in commentary.replace(". ", ".\n").split("\n") if s.strip()]
for i, sentence in enumerate(sentences, 1):
    print(f"  [{i}] {sentence}")
print("-" * 60)

# Chain visualisation
print("\nChain Summary:")
print(f"  Input:   {len(EARNINGS_TEXT):>6,} chars  (raw earnings text)")
print(f"  Step 1:  {len(json.dumps(extracted)):>6,} chars  (extracted JSON metrics)")
print(f"  Step 2:  {len(commentary):>6,} chars  (analyst commentary)")
print(f"  Compression ratio: {len(EARNINGS_TEXT) / len(commentary):.1f}x")


---
## Case Study 5 — AI Robo-Advisory Portfolio Dashboard

**Cross-reference:** Week 6 Seminar §5.1 | DDF Stations 1–4

This case study implements a complete AI-powered robo-advisory platform for retail wealth management. Students build and interact with a Streamlit dashboard that combines a five-question risk questionnaire with mean-variance portfolio optimisation and Cholesky Monte Carlo simulation to produce personalised investment recommendations, compliant with ASIC Regulatory Guide 255 (Digital Financial Product Advice).

| DDF Station | Robo-Adviser Implementation |
|---|---|
| **Station 1 (ETL)** | Asset universe parameters loaded; risk questionnaire responses ingested |
| **Station 2 (Feature Eng.)** | Risk score computed from 5 questionnaire items (max 100); ESG tilt applied to weights |
| **Station 3 (Model Design)** | Pre-optimised mean-variance weights per profile; Cholesky Monte Carlo (1,000 paths, 6 assets) |
| **Station 4 (Implementation)** | Streamlit dashboard: allocation pie, fan chart, metrics panel, ASIC RG 255 model card |

### Asset universe (6 Australian asset classes, calibrated 2000–2024)

| Asset Class | E[r] | σ | Notes |
|---|---|---|---|
| ASX 200 Equities | 9.3% | 15.5% | Domestic large-cap equities |
| Intl Equities (Hedged) | 8.6% | 14.0% | Currency-hedged global equities |
| Australian Bonds | 4.2% | 5.8% | AGS + semi-government bonds |
| Global Bonds (Hedged) | 3.8% | 6.5% | Bloomberg Global Aggregate (AUD hedged) |
| Australian REITs | 7.8% | 18.5% | ASX-listed property trusts |
| Cash (AUD) | 4.3% | 0.8% | RBA cash rate proxy |

### Regulatory anchors
- **ASIC RG 255** — digital financial product advice: disclosure, suitability, best interests
- **Corporations Act 2001 s961B** — best interests duty for personal advice
- **Privacy Act 1988 APPs 3/6** — collection and use of personal financial information

### Deliverables
1. Run the cell below to view inline simulation outputs (allocation pies + Monte Carlo fan chart)
2. Examine the portfolio statistics table across all four risk profiles
3. Launch the full Streamlit dashboard: `streamlit run robo_advisory_app.py`
4. *(Extension)* Add an efficient frontier plot with all four profiles marked; add a second tab with a batch portfolio comparison tool

In [ ]:
# Case Study 5: Write robo_advisory_app.py and run inline portfolio simulation.
# Writes the Streamlit robo-adviser to disk, then plots:
#   (1) allocation pie charts for all four risk profiles
#   (2) Monte Carlo fan chart for the Balanced profile (AUD 100k, 20yr horizon)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Asset universe and pre-optimised mean-variance weights ──
ASSET_NAMES = ['ASX 200 Equities', 'Intl Equities (Hedged)',
               'Australian Bonds', 'Global Bonds (Hedged)',
               'Australian REITs', 'Cash (AUD)']
MU    = np.array([0.093, 0.086, 0.042, 0.038, 0.078, 0.043])
SIGMA = np.array([0.155, 0.140, 0.058, 0.065, 0.185, 0.008])
CORR  = np.array([
    [1.00, 0.82, -0.18, -0.14,  0.58,  0.02],
    [0.82, 1.00, -0.12, -0.10,  0.48,  0.01],
    [-0.18,-0.12, 1.00,  0.84, -0.08,  0.12],
    [-0.14,-0.10, 0.84,  1.00, -0.06,  0.10],
    [0.58,  0.48, -0.08,-0.06,  1.00,  0.03],
    [0.02,  0.01,  0.12, 0.10,  0.03,  1.00],
])
COV = np.outer(SIGMA, SIGMA) * CORR

PROFILES = {
    'Conservative': np.array([0.10, 0.10, 0.35, 0.30, 0.05, 0.10]),
    'Balanced':     np.array([0.25, 0.20, 0.25, 0.15, 0.10, 0.05]),
    'Growth':       np.array([0.35, 0.28, 0.15, 0.10, 0.10, 0.02]),
    'High Growth':  np.array([0.45, 0.33, 0.05, 0.05, 0.10, 0.02]),
}
PROFILE_COLOURS = {'Conservative': '#1A7A4A', 'Balanced': '#003366',
                   'Growth': '#D4820A', 'High Growth': '#C8102E'}

def portfolio_stats(w):
    mu_p  = float(w @ MU)
    sig_p = float(np.sqrt(w @ COV @ w))
    sr    = (mu_p - 0.043) / sig_p
    return mu_p, sig_p, sr

def var_95_annual(w, V0=100_000):
    mu_p, sig_p, _ = portfolio_stats(w)
    return V0 * (1 - np.exp(mu_p - 1.645 * sig_p))

def monte_carlo_portfolio(w, V0, n_years, annual_contrib, n_sims=1000, seed=42):
    np.random.seed(seed)
    L = np.linalg.cholesky(COV)
    paths = np.zeros((n_sims, n_years + 1))
    paths[:, 0] = V0
    for t in range(n_years):
        z     = np.random.standard_normal((n_sims, len(MU)))
        r_all = z @ L.T + MU
        r_p   = r_all @ w
        paths[:, t + 1] = paths[:, t] * (1 + r_p) + annual_contrib
    return paths

def risk_score_to_profile(score):
    if score <= 25:  return 'Conservative'
    elif score <= 50: return 'Balanced'
    elif score <= 75: return 'Growth'
    else:             return 'High Growth'

# ── Inline simulation: profile allocation pies + fan chart ──
fig = plt.figure(figsize=(18, 9))
gs  = fig.add_gridspec(2, 4, hspace=0.45, wspace=0.30)

asset_colours = ['#003366','#C8102E','#1A7A4A','#6699CC','#D4820A','#888888']

# Row 1: allocation pies for all four profiles
for idx, (pname, w) in enumerate(PROFILES.items()):
    ax = fig.add_subplot(gs[0, idx])
    mu_p, sig_p, sr = portfolio_stats(w)
    wedge_labels = [f'{a}\n{v:.0%}' for a, v in zip(ASSET_NAMES, w)]
    wedges, _ = ax.pie(w, labels=None, colors=asset_colours,
                       startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 0.8})
    ax.set_title(f'{pname}\nE[r]={mu_p:.1%}  σ={sig_p:.1%}  SR={sr:.2f}',
                 fontsize=9, color=PROFILE_COLOURS[pname], fontweight='bold')

# Single legend for pies
handles = [mpatches.Patch(color=asset_colours[i], label=ASSET_NAMES[i])
           for i in range(len(ASSET_NAMES))]
fig.legend(handles=handles, loc='upper center', ncol=3, fontsize=8,
           bbox_to_anchor=(0.50, 0.56), framealpha=0.9)

# Row 2: Monte Carlo fan chart (Balanced profile, AUD 100k, 10k/yr contrib, 20yr)
ax_fan = fig.add_subplot(gs[1, :])
V0, n_years, annual_contrib = 100_000, 20, 10_000
paths_mc = monte_carlo_portfolio(PROFILES['Balanced'], V0, n_years, annual_contrib)
years_ax = np.arange(n_years + 1)
bands = np.percentile(paths_mc, [10, 25, 50, 75, 90], axis=0)

ax_fan.fill_between(years_ax, bands[0]/1e3, bands[4]/1e3, alpha=0.15, color='#003366', label='p10–p90')
ax_fan.fill_between(years_ax, bands[1]/1e3, bands[3]/1e3, alpha=0.30, color='#C8102E',  label='p25–p75')
ax_fan.plot(years_ax, bands[2]/1e3, color='#003366', lw=2.5, label='Median (p50)')
ax_fan.set_xlabel('Investment Horizon (years)', fontsize=10)
ax_fan.set_ylabel('Portfolio Value (AUD K)', fontsize=10)
ax_fan.set_title('CS5 Robo-Adviser: Monte Carlo Simulation — Balanced Profile\n'
                 f'Opening: AUD {V0/1e3:.0f}k | Annual contribution: AUD {annual_contrib/1e3:.0f}k | 1,000 paths, Cholesky decomposition',
                 color='#003366', fontsize=10)
ax_fan.legend(fontsize=9)

# Annotate p50 at year 20
p50_end = bands[2, -1] / 1e3
ax_fan.annotate(f'Median AUD {p50_end:.0f}k',
                xy=(n_years, bands[2, -1]/1e3), xytext=(n_years - 4, p50_end + 40),
                arrowprops=dict(arrowstyle='->', color='#003366'), fontsize=9, color='#003366')

plt.savefig('cs5_robo_inline_preview.png', bbox_inches='tight')
plt.show()

# Print metrics table
print("CS5 Robo-Adviser: Portfolio Statistics by Risk Profile")
print(f"{'Profile':<15} {'E[r]':>6} {'σ':>6} {'SR':>6} {'VaR 95% (AUD 100k)':>22}")
print("-" * 60)
for pname, w in PROFILES.items():
    mu_p, sig_p, sr = portfolio_stats(w)
    var = var_95_annual(w, 100_000)
    print(f"{pname:<15} {mu_p:>6.1%} {sig_p:>6.1%} {sr:>6.2f} {var:>22,.0f}")

# ── Write Streamlit app to disk ──
ROBO_APP_CODE = '''
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

st.set_page_config(page_title="FINS5557 Robo-Adviser | UNSW", layout="wide")

ASSET_NAMES = ["ASX 200 Equities","Intl Equities (Hedged)","Australian Bonds",
               "Global Bonds (Hedged)","Australian REITs","Cash (AUD)"]
MU    = np.array([0.093, 0.086, 0.042, 0.038, 0.078, 0.043])
SIGMA = np.array([0.155, 0.140, 0.058, 0.065, 0.185, 0.008])
CORR  = np.array([
    [1.00,0.82,-0.18,-0.14,0.58,0.02],[0.82,1.00,-0.12,-0.10,0.48,0.01],
    [-0.18,-0.12,1.00,0.84,-0.08,0.12],[-0.14,-0.10,0.84,1.00,-0.06,0.10],
    [0.58,0.48,-0.08,-0.06,1.00,0.03],[0.02,0.01,0.12,0.10,0.03,1.00],
])
COV = np.outer(SIGMA, SIGMA) * CORR
PROFILES = {
    "Conservative": np.array([0.10,0.10,0.35,0.30,0.05,0.10]),
    "Balanced":     np.array([0.25,0.20,0.25,0.15,0.10,0.05]),
    "Growth":       np.array([0.35,0.28,0.15,0.10,0.10,0.02]),
    "High Growth":  np.array([0.45,0.33,0.05,0.05,0.10,0.02]),
}
ASSET_COLOURS=["#003366","#C8102E","#1A7A4A","#6699CC","#D4820A","#888888"]
NAVY="#003366"; RED="#C8102E"; GREEN="#1A7A4A"

def portfolio_stats(w):
    mu_p=float(w@MU); sig_p=float(np.sqrt(w@COV@w)); sr=(mu_p-0.043)/sig_p
    return mu_p,sig_p,sr

def var_95(w,V0):
    mu_p,sig_p,_=portfolio_stats(w); return V0*(1-np.exp(mu_p-1.645*sig_p))

@st.cache_data
def monte_carlo(w_tuple,V0,n_years,contrib,n_sims=1000,seed=42):
    w=np.array(w_tuple); np.random.seed(seed); L=np.linalg.cholesky(COV)
    paths=np.zeros((n_sims,n_years+1)); paths[:,0]=V0
    for t in range(n_years):
        z=np.random.standard_normal((n_sims,len(MU))); r_all=z@L.T+MU
        paths[:,t+1]=paths[:,t]*(1+r_all@w)+contrib
    return paths

# ── Sidebar: risk questionnaire ──
st.sidebar.header("Risk Questionnaire")
q1=st.sidebar.radio("Investment objective?",["Capital preservation (1)","Income (2)","Balanced growth (3)","Capital growth (4)","Aggressive growth (5)"])
q2=st.sidebar.radio("Investment experience?",["None (1)","Limited (2)","Moderate (3)","Experienced (4)","Professional (5)"])
q3=st.sidebar.radio("Reaction to a 20% portfolio loss?",["Sell immediately (1)","Reduce exposure (2)","Hold (3)","Buy more (4)","Significantly increase (5)"])
q4=st.sidebar.radio("Investment horizon?",["< 2 years (1)","2–5 years (2)","5–10 years (3)","10–20 years (4)","> 20 years (5)"])
q5=st.sidebar.radio("Attitude to volatility?",["Strongly averse (1)","Somewhat averse (2)","Neutral (3)","Comfortable (4)","Prefer high vol (5)"])
score=sum(int(q.split("(")[1][0]) for q in [q1,q2,q3,q4,q5])*5

esg_on    = st.sidebar.toggle("ESG Tilt", value=False)
V0        = st.sidebar.number_input("Initial Investment (AUD)", 10_000, 5_000_000, 100_000, 10_000)
contrib   = st.sidebar.number_input("Annual Contribution (AUD)", 0, 200_000, 10_000, 1_000)
n_years   = st.sidebar.slider("Investment Horizon (years)", 1, 40, 20)

def score_to_profile(s):
    if s<=25: return "Conservative"
    elif s<=50: return "Balanced"
    elif s<=75: return "Growth"
    else: return "High Growth"

profile = score_to_profile(score)
w = PROFILES[profile].copy()
if esg_on:
    w[4]=max(w[4]*0.60,0.01); w[2]+=w[4]*0.40*0.60; w[3]+=w[4]*0.40*0.40
    w=w/w.sum()

mu_p,sig_p,sr=portfolio_stats(w); var95=var_95(w,V0)
paths=monte_carlo(tuple(w),V0,n_years,contrib)
bands=np.percentile(paths,[10,25,50,75,90],axis=0)
years_ax=np.arange(n_years+1)

# ── Main panel ──
st.title("FINS5557 Robo-Adviser: Portfolio Recommendation")
st.markdown("**UNSW Business School | Hric & Lin, 2026 | ASIC RG 255 Compliant**")
st.divider()

col1,col2,col3,col4=st.columns(4)
col1.metric("Risk Score",f"{score}/100",delta=profile)
col2.metric("Expected Return",f"{mu_p:.1%} p.a.")
col3.metric("Volatility (σ)",f"{sig_p:.1%} p.a.")
col4.metric("VaR 95% (1yr)",f"AUD {var95:,.0f}",delta=f"SR={sr:.2f}")
st.divider()

fig,axes=plt.subplots(1,2,figsize=(14,5))
# Allocation pie
axes[0].pie(w,labels=[f"{a}\\n{v:.0%}" for a,v in zip(ASSET_NAMES,w)],
            colors=ASSET_COLOURS,startangle=90,
            wedgeprops={"edgecolor":"white","linewidth":0.8})
axes[0].set_title(f"{profile} — Recommended Allocation",fontsize=10,color=NAVY)
# Fan chart
axes[1].fill_between(years_ax,bands[0]/1e3,bands[4]/1e3,alpha=0.15,color=NAVY,label="p10–p90")
axes[1].fill_between(years_ax,bands[1]/1e3,bands[3]/1e3,alpha=0.30,color=RED, label="p25–p75")
axes[1].plot(years_ax,bands[2]/1e3,color=NAVY,lw=2,label="Median")
axes[1].set_xlabel("Years"); axes[1].set_ylabel("Portfolio Value (AUD K)")
axes[1].set_title(f"Monte Carlo Projection ({n_years}yr | {1000} paths)"); axes[1].legend(fontsize=8)
st.pyplot(fig)

with st.expander("AI Robo-Adviser Model Card (ASIC RG 255)"):
    st.markdown(
        f"- **Profile assigned:** {profile} (score {score}/100)\\n"
        "- **Methodology:** mean-variance optimisation, Cholesky Monte Carlo (1,000 paths)\\n"
        "- **Asset universe:** 6 Australian asset classes, calibrated 2000-2024\\n"
        "- **Regulatory basis:** ASIC RG 255 (digital advice), Corporations Act 2001 s961B\\n"
        "- **ESG:** REITs weight reduced 40%; redirected to bonds when ESG tilt enabled\\n"
        "- **Limitations:** historical calibration; no tax, fees, or sequencing risk modelled\\n"
        "- **Disclosure:** this tool provides general advice only — not personal financial advice"
    )
'''

import os
with open('robo_advisory_app.py', 'w', encoding='utf-8') as f:
    f.write(ROBO_APP_CODE.lstrip('\n'))
print('\nrobo_advisory_app.py written to disk.')
print('\nRun the Robo-Adviser dashboard:')
print('  LOCAL:  streamlit run robo_advisory_app.py')
print('  COLAB:  !nohup streamlit run robo_advisory_app.py --server.port 8502 &')
print('          from pyngrok import ngrok; print(ngrok.connect(8502))')
print('\nTest scenarios:')
print('  Conservative:  q1=1, q2=1, q3=1, q4=1, q5=1  (score=25)')
print('  High Growth:   q1=5, q2=5, q3=5, q4=5, q5=5  (score=100)')
print('  Balanced+ESG:  score ~50, ESG toggle ON')

---
## Case Study 6 — Superannuation Retirement Fund Allocation Dashboard

**Cross-reference:** Week 6 Seminar §5.2 | DDF Stations 1–4

This case study implements a complete lifecycle superannuation planner for Australia's compulsory retirement savings system. The dashboard projects member balances to retirement using Monte Carlo simulation with an age-adaptive glide path, benchmarked against APRA SPS 515 member outcomes obligations and the ASFA Comfortable Retirement Standard.

| DDF Station | Superannuation Dashboard Implementation |
|---|---|
| **Station 1 (ETL)** | Member inputs ingested (age, salary, balance, contribution rates, investment option) |
| **Station 2 (Feature Eng.)** | Age-adjusted glide path computed each year; inflation-adjusted contribution stream |
| **Station 3 (Model Design)** | Monte Carlo (2,000 paths); lifecycle glide path reduces equity 0.8%/yr above age 50 |
| **Station 4 (Implementation)** | Streamlit planner with accumulation fan chart, glide path chart, sensitivity table, SPS 515 model card |

### Regulatory anchors
- **APRA SPS 515** — member outcomes assessment; trustee must demonstrate investment options deliver outcomes in members' best financial interests
- **APRA SPS 530** — investment governance; lifecycle strategy documentation
- **SIS Act 1993** — sole purpose test, trustee obligations
- **ASFA Retirement Standard** — comfortable retirement AUD 51,630/yr (single) / AUD 72,148/yr (couple), 2024–25

### Key financial mechanics
- **Superannuation Guarantee:** employer contributions at 11.5% of ordinary time earnings (2024–25); legislated to reach 12% by 2025–26
- **Concessional contributions tax:** 15% on employer and salary-sacrifice contributions — a 32pp advantage for a top-marginal-rate (47%) earner
- **Glide path rule:** equity allocation is reduced 0.8 percentage points per year above age 50, with a maximum 20pp total reduction — the standard lifecycle de-risking rule applied by major Australian superannuation funds

### Deliverables
1. Run the cell below to view an inline simulation preview (fan chart + glide path charts)
2. Examine the voluntary contribution sensitivity table
3. Launch the full Streamlit dashboard: `streamlit run super_fund_app.py`
4. *(Extension)* Add a third tab implementing the APRA SPS 515 member outcomes heat-map: compare four investment options by projected coverage ratio, P(shortfall), and PSI drift score

In [ ]:
# Case Study 6: Write super_fund_app.py and run inline accumulation simulation.
# Writes the Streamlit superannuation planner to disk, then plots:
#   (1) glide path allocation at five age milestones
#   (2) Monte Carlo accumulation fan chart for a 35-year-old Growth investor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Australian superannuation parameters (2024-25) ──
SG_RATE           = 0.115
CONTR_TAX         = 0.15
INFLATION         = 0.025
SAFE_WD_RATE      = 0.04
ASFA_COMFORTABLE_SINGLE = 51_630
ASFA_COMFORTABLE_COUPLE = 72_148

SUPER_ASSETS = ['Equities (Aust.)', 'Equities (Intl.)', 'Property', 'Bonds', 'Cash']
BASE_ALLOCS = {
    'Conservative': {'Equities (Aust.)': 0.10, 'Equities (Intl.)': 0.10,
                     'Property': 0.05, 'Bonds': 0.60, 'Cash': 0.15},
    'Balanced':     {'Equities (Aust.)': 0.25, 'Equities (Intl.)': 0.25,
                     'Property': 0.10, 'Bonds': 0.30, 'Cash': 0.10},
    'Growth':       {'Equities (Aust.)': 0.35, 'Equities (Intl.)': 0.30,
                     'Property': 0.15, 'Bonds': 0.15, 'Cash': 0.05},
    'High Growth':  {'Equities (Aust.)': 0.42, 'Equities (Intl.)': 0.38,
                     'Property': 0.10, 'Bonds': 0.08, 'Cash': 0.02},
}
ASSET_MU    = {'Equities (Aust.)': 0.088, 'Equities (Intl.)': 0.092,
               'Property': 0.074, 'Bonds': 0.040, 'Cash': 0.043}
ASSET_SIGMA = {'Equities (Aust.)': 0.155, 'Equities (Intl.)': 0.168,
               'Property': 0.135, 'Bonds': 0.058, 'Cash': 0.008}

def glide_path_alloc(age, profile):
    base = {k: v for k, v in BASE_ALLOCS[profile].items()}
    if age > 50:
        reduction = min((age - 50) * 0.008, 0.20)
        base['Equities (Aust.)'] = max(base['Equities (Aust.)'] - reduction * 0.5, 0.05)
        base['Equities (Intl.)'] = max(base['Equities (Intl.)'] - reduction * 0.5, 0.05)
        base['Bonds'] += reduction * 0.60
        base['Cash']  += reduction * 0.40
    total = sum(base.values())
    return {k: v / total for k, v in base.items()}

def simulate_accumulation(current_age, retire_age, balance, salary,
                           emp_pct, vol_pct, profile, n_sims=2000, seed=42):
    np.random.seed(seed)
    n_years = retire_age - current_age
    paths = np.zeros((n_sims, n_years + 1))
    paths[:, 0] = balance
    for t in range(n_years):
        age_t  = current_age + t
        alloc  = glide_path_alloc(age_t, profile)
        mu_t   = sum(alloc[a] * ASSET_MU[a]    for a in SUPER_ASSETS)
        sig_t  = sum(alloc[a] * ASSET_SIGMA[a] for a in SUPER_ASSETS) * 0.85
        r_t    = np.random.normal(mu_t, sig_t, n_sims)
        contr  = salary * (emp_pct + vol_pct) * (1 - CONTR_TAX) * (1 + INFLATION) ** t
        paths[:, t + 1] = paths[:, t] * (1 + r_t) + contr
    return paths, n_years

def sustainability_prob(median_balance, target_income, years=25, n_sims=2000, seed=99):
    np.random.seed(seed)
    wd = target_income
    count_ok = 0
    for _ in range(n_sims):
        bal = median_balance
        ok = True
        for _ in range(years):
            r = np.random.normal(0.05, 0.08)
            bal = bal * (1 + r) - wd
            wd *= (1 + INFLATION)
            if bal <= 0:
                ok = False; break
        count_ok += ok
    return count_ok / n_sims

# ── Inline simulation: glide path + accumulation fan chart ──
PROFILE_SIM = 'Growth'
current_age, retire_age = 35, 67
balance0 = 85_000
salary0  = 110_000
paths, n_years = simulate_accumulation(current_age, retire_age, balance0, salary0,
                                        SG_RATE, 0.0, PROFILE_SIM)
years_axis = np.arange(n_years + 1)
ages_axis  = current_age + years_axis

pcts   = [10, 25, 50, 75, 90]
bands  = np.percentile(paths, pcts, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel 1: glide path over age milestones
age_milestones = [35, 45, 55, 60, 65, 67]
x = np.arange(len(age_milestones))
width = 0.15
colours_super = [NAVY, RED, GREEN, ORANGE, PURPLE]
for j, asset in enumerate(SUPER_ASSETS):
    vals = [glide_path_alloc(a, PROFILE_SIM)[asset] * 100 for a in age_milestones]
    axes[0].bar(x + j * width, vals, width, label=asset, color=colours_super[j], alpha=0.85)
axes[0].set_xticks(x + width * 2)
axes[0].set_xticklabels([f'Age {a}' for a in age_milestones], fontsize=9)
axes[0].set_ylabel('Allocation (%)')
axes[0].set_title(f'Glide Path — {PROFILE_SIM} Option (APRA SPS 530)', color=NAVY, fontsize=11)
axes[0].legend(fontsize=8, loc='upper right')
axes[0].axvline(4.6, color='red', linestyle=':', lw=1.2, label='_Age 65')
axes[0].text(4.65, axes[0].get_ylim()[1] * 0.85, 'retire', fontsize=7, color='red')

# Panel 2: Monte Carlo fan chart
fill_colours = ['#A8C8F0', '#6699CC', '#003366', '#6699CC', '#A8C8F0']
labels_fan   = ['p10–p90', 'p25–p75', 'Median (p50)', None, None]
for i in range(4):
    axes[1].fill_between(ages_axis, bands[i] / 1e6, bands[4 - i] / 1e6,
                         alpha=0.25, color=fill_colours[i])
axes[1].plot(ages_axis, bands[2] / 1e6, color=NAVY, lw=2.5, label='Median (p50)')
axes[1].fill_between(ages_axis, bands[1] / 1e6, bands[3] / 1e6, alpha=0.30, color=RED,
                     label='p25–p75')
axes[1].axhline(ASFA_COMFORTABLE_SINGLE * 25 / 1e6, color=GREEN, linestyle='--',
                lw=1.5, label=f'ASFA target (25yr × AUD {ASFA_COMFORTABLE_SINGLE/1e3:.0f}k)')
axes[1].set_xlabel('Age'); axes[1].set_ylabel('Projected Balance (AUD M)')
axes[1].set_title(f'Accumulation Simulation — {PROFILE_SIM}, Age {current_age}→{retire_age}\n'
                  f'Balance: AUD {balance0/1e3:.0f}k | Salary: AUD {salary0/1e3:.0f}k | SG: {SG_RATE:.1%}',
                  color=NAVY, fontsize=10)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('cs6_super_inline_preview.png', bbox_inches='tight')
plt.show()

median_retire = np.median(paths[:, -1])
prob_ok       = sustainability_prob(median_retire, ASFA_COMFORTABLE_SINGLE)
print(f"\nCS6 Inline Simulation — {PROFILE_SIM} (Age {current_age}→{retire_age})")
print(f"  Opening balance:   AUD {balance0:>10,.0f}")
print(f"  Median balance at retirement: AUD {median_retire:>10,.0f}")
print(f"  p10 / p90:         AUD {np.percentile(paths[:,-1],10):>10,.0f} / AUD {np.percentile(paths[:,-1],90):>10,.0f}")
print(f"  P(fund lasts 25yr, ASFA comfortable income): {prob_ok:.1%}")
print(f"  APRA SPS 515 benchmark: {ASFA_COMFORTABLE_SINGLE:,}/yr (ASFA comfortable, single, 2024-25)")

# Sensitivity: voluntary contribution impact
print(f"\nVoluntary Contribution Sensitivity (median retirement balance):")
print(f"  {'Extra Contrib':>14} | {'Median Balance':>16} | {'vs Base':>10}")
print("  " + "-" * 45)
base_median = np.median(simulate_accumulation(current_age, retire_age, balance0, salary0,
                                               SG_RATE, 0.0, PROFILE_SIM)[0][:, -1])
for extra in [0.00, 0.02, 0.05, 0.08, 0.10]:
    m = np.median(simulate_accumulation(current_age, retire_age, balance0, salary0,
                                         SG_RATE, extra, PROFILE_SIM)[0][:, -1])
    diff = m - base_median
    print(f"  {extra:>14.0%} | AUD {m:>12,.0f} | +AUD {diff:>7,.0f}")

# ── Write Streamlit app to disk ──
SUPER_APP_CODE = '''
import streamlit as st
import numpy as np
import matplotlib.pyplot as plt

st.set_page_config(page_title="FINS5557 Super Fund Planner | UNSW", layout="wide")

SG_RATE = 0.115; CONTR_TAX = 0.15; INFLATION = 0.025
ASFA_COMFORTABLE_SINGLE = 51_630; ASFA_COMFORTABLE_COUPLE = 72_148
SUPER_ASSETS = ["Equities (Aust.)", "Equities (Intl.)", "Property", "Bonds", "Cash"]
BASE_ALLOCS = {
    "Conservative": {"Equities (Aust.)": 0.10, "Equities (Intl.)": 0.10,
                     "Property": 0.05, "Bonds": 0.60, "Cash": 0.15},
    "Balanced":     {"Equities (Aust.)": 0.25, "Equities (Intl.)": 0.25,
                     "Property": 0.10, "Bonds": 0.30, "Cash": 0.10},
    "Growth":       {"Equities (Aust.)": 0.35, "Equities (Intl.)": 0.30,
                     "Property": 0.15, "Bonds": 0.15, "Cash": 0.05},
    "High Growth":  {"Equities (Aust.)": 0.42, "Equities (Intl.)": 0.38,
                     "Property": 0.10, "Bonds": 0.08, "Cash": 0.02},
}
ASSET_MU    = {"Equities (Aust.)": 0.088, "Equities (Intl.)": 0.092,
               "Property": 0.074, "Bonds": 0.040, "Cash": 0.043}
ASSET_SIGMA = {"Equities (Aust.)": 0.155, "Equities (Intl.)": 0.168,
               "Property": 0.135, "Bonds": 0.058, "Cash": 0.008}
NAVY="#003366"; RED="#C8102E"; GREEN="#1A7A4A"; ORANGE="#D4820A"; PURPLE="#6B21A8"
COLOURS=[NAVY,RED,GREEN,ORANGE,PURPLE]

def glide_path_alloc(age, profile):
    base = {k: v for k, v in BASE_ALLOCS[profile].items()}
    if age > 50:
        reduction = min((age - 50) * 0.008, 0.20)
        base["Equities (Aust.)"] = max(base["Equities (Aust.)"] - reduction * 0.5, 0.05)
        base["Equities (Intl.)"] = max(base["Equities (Intl.)"] - reduction * 0.5, 0.05)
        base["Bonds"] += reduction * 0.60; base["Cash"] += reduction * 0.40
    total = sum(base.values())
    return {k: v/total for k, v in base.items()}

@st.cache_data
def simulate_accumulation(current_age, retire_age, balance, salary, emp_pct, vol_pct, profile, seed=42):
    np.random.seed(seed); n_years = retire_age - current_age
    paths = np.zeros((2000, n_years+1)); paths[:,0] = balance
    for t in range(n_years):
        alloc = glide_path_alloc(current_age+t, profile)
        mu_t  = sum(alloc[a]*ASSET_MU[a] for a in SUPER_ASSETS)
        sig_t = sum(alloc[a]*ASSET_SIGMA[a] for a in SUPER_ASSETS)*0.85
        r_t   = np.random.normal(mu_t, sig_t, 2000)
        contr = salary*(emp_pct+vol_pct)*(1-CONTR_TAX)*(1+INFLATION)**t
        paths[:,t+1] = paths[:,t]*(1+r_t)+contr
    return paths

# ── Sidebar ──
st.sidebar.header("Member Profile")
current_age     = st.sidebar.slider("Current Age", 20, 60, 35)
retire_age      = st.sidebar.slider("Target Retirement Age", 55, 70, 67)
current_balance = st.sidebar.number_input("Current Super Balance (AUD)", 0, 2_000_000, 85_000, 5_000)
salary          = st.sidebar.number_input("Annual Salary (AUD)", 30_000, 500_000, 110_000, 5_000)
vol_pct         = st.sidebar.slider("Extra Voluntary Contributions (%)", 0.0, 0.15, 0.0, 0.01)
profile         = st.sidebar.selectbox("Investment Option", list(BASE_ALLOCS.keys()), index=2)
couple          = st.sidebar.checkbox("Planning for a couple?", value=False)
asfa_target     = ASFA_COMFORTABLE_COUPLE if couple else ASFA_COMFORTABLE_SINGLE

paths = simulate_accumulation(current_age, retire_age, current_balance,
                               salary, SG_RATE, vol_pct, profile)
n_years = retire_age - current_age
ages = np.arange(current_age, retire_age+1)
p10,p25,p50,p75,p90 = np.percentile(paths,[10,25,50,75,90],axis=0)
median_retire = np.median(paths[:,-1])
withdrawal    = asfa_target * (1+INFLATION)**(retire_age-current_age)
coverage_ratio = median_retire*0.04 / withdrawal

# ── Main panel ──
st.title("FINS5557 Superannuation Retirement Planner")
st.markdown("**UNSW Business School | Hric & Lin, 2026 | APRA SPS 515/530 Compliant**")
st.divider()

col1,col2,col3,col4 = st.columns(4)
col1.metric("Projected Balance (Median)", f"AUD {median_retire/1e6:.2f}M")
col2.metric("ASFA Comfortable Target", f"AUD {asfa_target:,}/yr")
col3.metric("Safe Withdrawal Rate", "4.0% p.a.")
col4.metric("Coverage Ratio", f"{coverage_ratio:.2f}x",
            delta=f"{'Surplus' if coverage_ratio>=1 else 'Shortfall'}")

st.divider()

fig, axes = plt.subplots(1,2,figsize=(14,5))
# Fan chart
axes[0].fill_between(ages,p10/1e6,p90/1e6,alpha=0.15,color=NAVY,label="p10–p90")
axes[0].fill_between(ages,p25/1e6,p75/1e6,alpha=0.30,color=RED, label="p25–p75")
axes[0].plot(ages,p50/1e6,color=NAVY,lw=2,label="Median")
axes[0].axhline(asfa_target*25/1e6,color=GREEN,linestyle="--",lw=1.5,
                label=f"ASFA 25yr: AUD {asfa_target*25/1e6:.2f}M")
axes[0].set_xlabel("Age"); axes[0].set_ylabel("Balance (AUD M)")
axes[0].set_title(f"Accumulation Projection — {profile}"); axes[0].legend(fontsize=8)
# Glide path
milestones=[current_age]+list(range(max(current_age+5,50),retire_age+1,5))[:5]+[retire_age]
milestones=sorted(set(milestones))
x=np.arange(len(milestones)); w=0.16
for j,asset in enumerate(SUPER_ASSETS):
    vals=[glide_path_alloc(a,profile)[asset]*100 for a in milestones]
    axes[1].bar(x+j*w,vals,w,label=asset,color=COLOURS[j],alpha=0.85)
axes[1].set_xticks(x+w*2); axes[1].set_xticklabels([f"Age {a}" for a in milestones],fontsize=8,rotation=15)
axes[1].set_ylabel("Allocation (%)"); axes[1].set_title(f"Lifecycle Glide Path — {profile}")
axes[1].legend(fontsize=7)
st.pyplot(fig)

# Sensitivity table
rows=[]
for extra in [0.0,0.02,0.05,0.08,0.10,0.15]:
    p=simulate_accumulation(current_age,retire_age,current_balance,salary,SG_RATE,extra,profile)
    m=np.median(p[:,-1]); rows.append({"Extra Contribution":f"{extra:.0%}","Median Balance":f"AUD {m/1e6:.3f}M","Coverage Ratio":f"{m*0.04/withdrawal:.2f}x"})
st.subheader("Voluntary Contribution Sensitivity")
st.dataframe(pd.DataFrame(rows),hide_index=True)

with st.expander("APRA SPS 515 Model Card"):
    st.markdown(
        "- **Regulatory basis:** APRA SPS 515 (member outcomes), SPS 530 (investment governance), SIS Act 1993\\n"
        "- **Methodology:** Monte Carlo (2,000 paths), Cholesky-free normal approximation, lifecycle glide path\\n"
        "- **Glide path rule:** equity allocation reduces 0.8%/yr above age 50, max 20pp total reduction\\n"
        "- **ASFA benchmark:** comfortable retirement AUD 51,630/yr (single, 2024-25)\\n"
        "- **Limitations:** assumes normal return distribution; excludes death/disability, tax optimisation\\n"
        "- **Data:** synthetic projections for educational purposes only — not personal financial advice"
    )
'''

import os
with open('super_fund_app.py', 'w', encoding='utf-8') as f:
    f.write(SUPER_APP_CODE.lstrip('\n'))
print('\nsuper_fund_app.py written to disk.')
print('\nRun the Superannuation Planner dashboard:')
print('  LOCAL:  streamlit run super_fund_app.py')
print('  COLAB:  !nohup streamlit run super_fund_app.py --server.port 8503 &')
print('          from pyngrok import ngrok; print(ngrok.connect(8503))')
print('\nTest scenarios:')
print('  Scenario A: Age 45, AUD 150k balance, AUD 130k salary, Growth option, 5% extra')
print('  Scenario B: Age 55, AUD 400k balance, AUD 160k salary, Balanced option, 0% extra')
print('  Scenario C: Age 28, AUD 15k  balance, AUD 80k  salary, High Growth option')


---
## Week 6 Summary

| Challenge | Key deliverable | Core skill demonstrated |
|---|---|---|
| C1 | Credit risk Pipeline | sklearn Pipeline, SHAP adverse action notice |
| C2 | LLM earnings extractor | Prompt engineering, multi-provider JSON extraction |
| C3 | Momentum backtest | Signal generation, Sharpe/MDD/CR/AR evaluation |
| C4 | Streamlit dashboard | Rapid UI, model serving, SR 11-7 model card |
| E1 | Prompt chaining | 2-step LLM workflow, token-efficient extraction |

**Key take-aways:**
- The `sklearn.Pipeline` is the production pattern for ML in finance — it eliminates data leakage and satisfies SR 11-7 traceability requirements
- Multi-provider LLM abstraction (`call_llm()`) decouples application logic from vendor dependency, addressing APRA CPS 230 vendor lock-in risk
- Prompt engineering (role + task + format + constraints) reliably produces structured JSON from unstructured financial documents without post-processing heuristics
- Backtesting requires `shift(1)` on signals to prevent look-ahead bias — omitting this is one of the most common research errors in quantitative finance
- Streamlit enables production-quality demos in <100 lines of Python; the Model Card pattern translates directly into SR 11-7 documentation artefacts

**Week 7 preview:** Industry applications begin — Week 7 covers AI in asset management and algorithmic trading, building on the sklearn pipelines, backtesting infrastructure, and LLM integrations established this week.

---

## References

Anthropic (2025). *API documentation*. https://docs.anthropic.com

Google (2025). *Gemini API documentation*. https://ai.google.dev

Hric, J., & Lin, Y. (2026). *FINS5557 applied AI in finance course notes: Week 6 — tech lab*. UNSW Business School.

McKinney, W. (2022). *Python for data analysis* (3rd ed.). O'Reilly.

OpenAI (2025). *Platform documentation*. https://platform.openai.com/docs

Pedregosa, F., et al. (2011). Scikit-learn: Machine learning in Python. *JMLR*, 12, 2825–2830.

Streamlit Inc. (2025). *Streamlit documentation*. https://docs.streamlit.io
